In [1]:
import os
from dotenv import load_dotenv
load_dotenv() # loading all environment variables

groq_api_key = os.getenv('GROQ_API_KEY')

In [2]:
from langchain_groq import ChatGroq

model = ChatGroq(model='llama-3.3-70b-versatile', groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x7c26bf7786e0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7c26bf779400>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage  # Tells that user is giving the message
model.invoke([HumanMessage(content = 'Hi, My name is Vibhav and I am working as a Data Analyst Intern')])

AIMessage(content="Hello Vibhav, nice to meet you. Congratulations on your Data Analyst Intern role. How's your experience been so far? What kind of projects have you been working on, and what tools or technologies are you using in your internship?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 52, 'total_tokens': 101, 'completion_time': 0.13959148, 'completion_tokens_details': None, 'prompt_time': 0.004737052, 'prompt_tokens_details': None, 'queue_time': 0.165481978, 'total_time': 0.144328532}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eef2d-eada-7e60-b1b2-abd99c405e51-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 49, 'total_tokens': 101})

In [4]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage('Hi, My name is Vibhav and I am working as a Data Analyst Intern'),
        AIMessage(content="Hello Vibhav, nice to meet you! Congratulations on your Data Analyst Intern role. How's your experience been so far? Are you enjoying the work and learning new things? What kind of projects are you working on, if you don't mind me asking?"),
        HumanMessage(content="Hey, What is my name and What do i do ?")
    ]
)

AIMessage(content="Your name is Vibhav, and you're working as a Data Analyst Intern.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 127, 'total_tokens': 145, 'completion_time': 0.039291504, 'completion_tokens_details': None, 'prompt_time': 0.018161468, 'prompt_tokens_details': None, 'queue_time': 0.052172352, 'total_time': 0.057452972}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eef2d-ec8b-7ca0-a2c9-40486dc810e5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 127, 'output_tokens': 18, 'total_tokens': 145})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id:str)-> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [6]:
config = {'configurable':{'session_id':'chat01'}}

In [7]:
response = with_message_history.invoke(
    [HumanMessage(content = "Hi, My name is Vibhav and I am working as a Data Analyst Intern")
        ],
        config=config
)

In [8]:
response.content

"Hello Vibhav, nice to meet you. Congratulations on your Data Analyst Intern position. How's your experience been so far? What kind of projects are you working on, and what tools or technologies are you using in your role?"

In [9]:
with_message_history.invoke(
    [HumanMessage(content="What's my name ?")],
    config=config
)

AIMessage(content='Your name is Vibhav.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 114, 'total_tokens': 122, 'completion_time': 0.019969432, 'completion_tokens_details': None, 'prompt_time': 0.019368287, 'prompt_tokens_details': None, 'queue_time': 0.162323263, 'total_time': 0.039337719}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eef2d-eef9-7f43-bd2b-d134580a8add-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 114, 'output_tokens': 8, 'total_tokens': 122})

In [10]:
# Change the config -> session
config02 = {'configurable':{'session_id':'chat02'}}

with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config = config02
)

AIMessage(content="I don't know your name. I'm a large language model, I don't have any information about you, including your name. I'm here to help answer your questions and provide information, but I don't have any personal data about you. If you'd like to share your name, I'd be happy to chat with you!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 40, 'total_tokens': 109, 'completion_time': 0.14311353, 'completion_tokens_details': None, 'prompt_time': 0.001751218, 'prompt_tokens_details': None, 'queue_time': 0.054038542, 'total_time': 0.144864748}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eef2d-effb-72a2-8f1d-164e7e26d473-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tokens': 69, 'total_tokens': 109})

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [11]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate(
    [
        ("system","You are a helpful assistent. Answer all the questionto the best of your abilities"),
        MessagesPlaceholder(variable_name="messages")   # an message will be given in a key value pair where key is "Messages"
     ]  
)

chain = prompt|model

In [12]:
chain.invoke({"messages":[HumanMessage(content="Hello, my name is Vibhav")]})

AIMessage(content="Hello Vibhav, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 60, 'total_tokens': 88, 'completion_time': 0.042240095, 'completion_tokens_details': None, 'prompt_time': 0.002952675, 'prompt_tokens_details': None, 'queue_time': 0.160505975, 'total_time': 0.04519277}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eef2d-f172-7370-b6c0-e77d7108856b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 28, 'total_tokens': 88})

In [13]:
# Combinnig Message History and Message Placeholder
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [14]:
confi= {"configurable": {"session_id": "chat03"}}
response = with_message_history.invoke(
    [HumanMessage(content="Hello, my name is Vibhav")],
    config = config
)

response

AIMessage(content='Hello again Vibhav, nice to chat with you. How can I assist you today? Do you have any questions or need help with something related to your Data Analyst Intern role?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 156, 'total_tokens': 194, 'completion_time': 0.133072649, 'completion_tokens_details': None, 'prompt_time': 0.007831465, 'prompt_tokens_details': None, 'queue_time': 0.161815524, 'total_time': 0.140904114}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eef2d-f288-7241-955a-9a2b0c7e4f0e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 156, 'output_tokens': 38, 'total_tokens': 194})

In [15]:
# Adding more complexity
prompt = ChatPromptTemplate(
    [
        (
            "system",
            "You are a helpful assistant. Answer all the question in the best of your ability in {language}"
        ),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt|model

In [16]:
response = chain.invoke(
    {
        "messages":[HumanMessage(content="My name is Vibhav")],
        "language":"Hindi"
        })

response.content

'नमस्ते विभाव, मैं आपकी कैसे मदद कर सकता हूँ?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [17]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [18]:
config = {"configurable":{"session_id":"chat_04"}}
response = with_message_history.invoke(
    {
        "messages":[HumanMessage(content="Hi, I am Vibhav")],
        "language": "Hindi"
    },
    config = config
)
response.content

'नमस्ते विभाव, मैं आपकी मदद करने में खुश हूँ। क्या मैं आपकी किसी प्रश्न या समस्या में सहायता कर सकता हूँ?'

In [19]:
response = with_message_history.invoke(
    {
        "messages":[HumanMessage(content="What is my name?")],
        "language": "Hindi"
    },
    config = config
)
response.content

'आपका नाम विभाव है।'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [20]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens =35,
    strategy = "last",
    token_counter = model,
    include_system = True,
    allow_partial = False,
    start_on = "human"
)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

/home/vibhav-ucharia/PyVenv/.venv/lib/python3.13/site-packages/langchain_core/language_models/base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


: 

In [ ]:
# Importing trimmer in chains
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough(messages = itemgetter("messages") | trimmer)
    | prompt
    |model
)

response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="What ice cream do i like ?")],
        "language": "English"
    }
)

response.content

In [ ]:
# Let's map it message history

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

config = {"configurable":{"session_id":"chat_05"}}

In [ ]:
response = with_message_history.invoke(
    {
    "messages": messages + [HumanMessage(content="What's my name ?")],
    "language": "English"
    },
    config = config
)

response.content

'Your name is Bob.'